In [ ]:
# Install dependencies
%pip install -q timm>=0.9 pytorch-metric-learning faiss-cpu scikit-learn

import json
import random
import time
import urllib.request
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image
from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset, Sampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({props.total_memory / 1024**3:.1f} GB)")
    torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path("/kaggle/working/09g_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "img_size": (256, 256),
    "batch_p": 16,
    "batch_k": 4,
    "eval_batch_size": 128,
    "num_workers": 4,
    "train_epochs": 150,
    "eval_every": 10,
    "lr": 1.0e-3,
    "backbone_lr_factor": 0.1,
    "eta_min": 1.0e-6,
    "weight_decay": 5.0e-4,
    "warmup_epochs": 10,
    "warmup_start_factor": 0.1,
    "label_smoothing": 0.05,
    "triplet_margin": 0.3,
    "circle_weight": 0.0,
    "cam_loss_weight": 0.3,
    "dmt_warmup_epochs": 15,
    "grl_lambda_max": 1.0,
    "gem_p": 3.0,
    "gradient_clip_norm": 5.0,
    "fp16": True,
    "use_flip_eval": True,
}

TRAIN_BATCH_SIZE = CFG["batch_p"] * CFG["batch_k"]
print(
    json.dumps(
        {
            **CFG,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "pretraining": "ImageNet IBN-Net official weights",
            "resume_from_09d_checkpoint": False,
        },
        indent=2,
    )
)

In [ ]:
import cv2

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
EXPECTED_CITYFLOW_DATASET_ROOT = KAGGLE_INPUT_ROOT / "data-aicity-2023-track-2"


def dedupe_paths(paths):
    unique = []
    seen = set()
    for path in paths:
        text = str(path)
        if text in seen:
            continue
        seen.add(text)
        unique.append(path)
    return unique


def find_named_paths(name, want_dir=None, max_results=50):
    if not KAGGLE_INPUT_ROOT.exists():
        return []
    matches = []
    for path in KAGGLE_INPUT_ROOT.rglob(name):
        if want_dir is True and not path.is_dir():
            continue
        if want_dir is False and not path.is_file():
            continue
        matches.append(path)
        if len(matches) >= max_results:
            break
    return sorted(matches)


def resolve_cityflow_root():
    candidate_bases = dedupe_paths(
        [EXPECTED_CITYFLOW_DATASET_ROOT]
        + find_named_paths("data-aicity-2023-track-2", want_dir=True, max_results=20)
    )
    for base in candidate_bases:
        if base.exists():
            print(f"[cityflow] using dataset root: {base}")
            return base

    nested_roots = find_named_paths("AIC22_Track1_MTMC_Tracking", want_dir=True, max_results=20)
    if nested_roots:
        root = nested_roots[0].parent
        print(f"[cityflow] inferred dataset root from nested directory: {root}")
        return root

    raise FileNotFoundError(
        "CityFlowV2 raw dataset not found anywhere under /kaggle/input. "
        "Expected dataset slug: data-aicity-2023-track-2"
    )


def resolve_raw_split_roots(cityflow_root):
    candidates = []
    for subpath in (
        "AIC22_Track1_MTMC_Tracking/train",
        "AIC22_Track1_MTMC_Tracking/validation",
        "train",
        "validation",
    ):
        candidate = cityflow_root / subpath
        if candidate.exists():
            candidates.append(candidate)

    candidates = dedupe_paths(candidates)
    if candidates:
        print("[cityflow] raw split roots:")
        for candidate in candidates:
            print(f"  - {candidate}")
        return candidates

    inferred_roots = []
    for gt_path in find_named_paths("gt.txt", want_dir=False, max_results=200):
        if gt_path.parent.name != "gt":
            continue
        camera_dir = gt_path.parent.parent
        scene_dir = camera_dir.parent
        if not camera_dir.name.startswith("c") or not scene_dir.name.startswith("S"):
            continue
        if not (camera_dir / "vdo.avi").exists():
            continue
        inferred_roots.append(scene_dir.parent)

    inferred_roots = dedupe_paths(inferred_roots)
    if inferred_roots:
        print("[cityflow] inferred split roots from gt/video scan:")
        for candidate in inferred_roots:
            print(f"  - {candidate}")
        return inferred_roots

    raise FileNotFoundError(
        "CityFlowV2 raw train/validation roots not found under /kaggle/input."
    )


CITYFLOW_ROOT = resolve_cityflow_root()
RAW_SPLIT_ROOTS = resolve_raw_split_roots(CITYFLOW_ROOT)

EXTRACTION_CFG = {
    "sample_stride": 5,
    "max_crops_per_id_cam": 24,
    "min_area": 2000,
    "min_bbox_side": 30,
    "train_ratio": 0.70,
    "min_cameras_for_eval": 2,
}

CROP_DIR = OUTPUT_DIR / "cityflowv2_crops"
CROP_DIR.mkdir(parents=True, exist_ok=True)
CROP_MANIFEST_PATH = OUTPUT_DIR / "cityflowv2_crops_manifest.json"


def camera_key(camera_dir: Path) -> str:
    return f"{camera_dir.parent.name}_{camera_dir.name}"


camera_dirs = sorted(
    camera_dir
    for split_root in RAW_SPLIT_ROOTS
    for camera_dir in split_root.glob("S*/c*")
    if camera_dir.is_dir()
)
if not camera_dirs:
    raise RuntimeError(f"No camera directories found under {RAW_SPLIT_ROOTS}")

CAMERA_DIRS = {camera_key(camera_dir): camera_dir for camera_dir in camera_dirs}
CAMERA_NAMES = sorted(CAMERA_DIRS)


def load_gt_rows(gt_path: Path) -> list[tuple[int, int, int, int, int, int]]:
    rows = []
    with gt_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            parts = raw_line.strip().split(",")
            if len(parts) < 6:
                continue
            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))
            x = int(float(parts[2]))
            y = int(float(parts[3]))
            w = int(float(parts[4]))
            h = int(float(parts[5]))
            confidence = float(parts[6]) if len(parts) > 6 else 1.0
            if track_id <= 0 or confidence <= 0:
                continue
            rows.append((frame_id, track_id, x, y, w, h))
    return rows


def sample_track_detections(detections):
    ordered = sorted(detections, key=lambda item: item[0])
    sampled = ordered[:: EXTRACTION_CFG["sample_stride"]]
    if ordered and sampled and sampled[-1][0] != ordered[-1][0]:
        sampled.append(ordered[-1])
    max_crops = EXTRACTION_CFG["max_crops_per_id_cam"]
    if max_crops and len(sampled) > max_crops:
        keep = np.linspace(0, len(sampled) - 1, num=max_crops, dtype=int)
        sampled = [sampled[index] for index in keep.tolist()]
    return sampled


def extract_camera_crops(cam_name: str, cam_dir: Path) -> dict[int, list[dict]]:
    gt_path = cam_dir / "gt" / "gt.txt"
    video_path = cam_dir / "vdo.avi"
    if not gt_path.exists() or not video_path.exists():
        print(f"Skipping {cam_name}: missing {gt_path.name if not gt_path.exists() else video_path.name}")
        return {}

    detections = defaultdict(list)
    for frame_id, track_id, x, y, w, h in load_gt_rows(gt_path):
        if w * h < EXTRACTION_CFG["min_area"]:
            continue
        if min(w, h) < EXTRACTION_CFG["min_bbox_side"]:
            continue
        detections[track_id].append((frame_id, x, y, w, h))

    sampled_by_frame = defaultdict(list)
    for track_id, track_detections in detections.items():
        for frame_id, x, y, w, h in sample_track_detections(track_detections):
            sampled_by_frame[frame_id].append((track_id, x, y, w, h))

    if not sampled_by_frame:
        print(f"Skipping {cam_name}: no detections after filtering")
        return {}

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Skipping {cam_name}: failed to open {video_path}")
        return {}

    records_by_track = defaultdict(list)
    written = 0
    for frame_id in sorted(sampled_by_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(frame_id - 1, 0))
        ok, frame = cap.read()
        if not ok:
            continue
        frame_h, frame_w = frame.shape[:2]
        for track_id, x, y, w, h in sampled_by_frame[frame_id]:
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame_w, x + w)
            y2 = min(frame_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            crop_path = CROP_DIR / f"{track_id:04d}_{cam_name}_f{frame_id:06d}.jpg"
            if not crop_path.exists() and not cv2.imwrite(str(crop_path), crop):
                continue
            records_by_track[track_id].append(
                {
                    "path": str(crop_path),
                    "pid": int(track_id),
                    "camname": cam_name,
                    "frame_id": int(frame_id),
                }
            )
            written += 1

    cap.release()
    print(f"{cam_name}: wrote {written} crops across {len(records_by_track)} vehicle IDs")
    return {track_id: sorted(items, key=lambda item: item["frame_id"]) for track_id, items in records_by_track.items()}


def manifest_is_usable(payload: dict) -> bool:
    if not CROP_DIR.exists():
        return False
    if payload.get("crop_dir") != str(CROP_DIR):
        return False
    return any(CROP_DIR.glob("*.jpg"))


all_crops = None
if CROP_MANIFEST_PATH.exists():
    with CROP_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        manifest_payload = json.load(handle)
    if manifest_is_usable(manifest_payload):
        all_crops = {
            int(pid): {
                camname: [dict(record) for record in records]
                for camname, records in camera_records.items()
            }
            for pid, camera_records in manifest_payload["all_crops"].items()
        }
        print(f"Reusing cached crops from {CROP_DIR}")

if all_crops is None:
    for existing_crop in CROP_DIR.glob("*.jpg"):
        existing_crop.unlink()
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam_name in CAMERA_NAMES:
        extracted = extract_camera_crops(cam_name, CAMERA_DIRS[cam_name])
        for pid, records in extracted.items():
            all_crops[pid][cam_name].extend(records)
    all_crops = {
        int(pid): {camname: sorted(records, key=lambda item: item["frame_id"]) for camname, records in camera_records.items()}
        for pid, camera_records in all_crops.items()
    }
    with CROP_MANIFEST_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "cityflow_root": str(CITYFLOW_ROOT),
                "raw_split_roots": [str(path) for path in RAW_SPLIT_ROOTS],
                "crop_dir": str(CROP_DIR),
                "all_crops": all_crops,
            },
            handle,
            indent=2,
        )

if not all_crops:
    raise RuntimeError("No crops were extracted from CityFlowV2 raw videos")

multi_cam_ids = sorted(
    pid for pid, camera_records in all_crops.items() if len(camera_records) >= EXTRACTION_CFG["min_cameras_for_eval"]
)
single_cam_ids = sorted(pid for pid, camera_records in all_crops.items() if len(camera_records) < EXTRACTION_CFG["min_cameras_for_eval"])
if len(multi_cam_ids) < 2:
    raise RuntimeError(
        f"Need at least 2 multi-camera vehicle IDs for train/eval splitting, found {len(multi_cam_ids)}"
    )

rng = np.random.default_rng(SEED)
shuffled_multi_cam_ids = multi_cam_ids[:]
rng.shuffle(shuffled_multi_cam_ids)
n_train_ids = int(round(len(shuffled_multi_cam_ids) * EXTRACTION_CFG["train_ratio"]))
n_train_ids = min(max(n_train_ids, 1), len(shuffled_multi_cam_ids) - 1)
train_identity_ids = set(shuffled_multi_cam_ids[:n_train_ids])
eval_identity_ids = set(shuffled_multi_cam_ids[n_train_ids:])

train_records = []
query_records = []
gallery_records = []

for pid in sorted(train_identity_ids):
    for camname, records in sorted(all_crops[pid].items()):
        for record in records:
            train_records.append(dict(record))

for pid in sorted(eval_identity_ids):
    for camname, records in sorted(all_crops[pid].items()):
        ordered_records = sorted(records, key=lambda item: (item["frame_id"], item["path"]))
        if not ordered_records:
            continue
        query_index = int(rng.integers(0, len(ordered_records)))
        query_records.append(dict(ordered_records[query_index]))
        for index, record in enumerate(ordered_records):
            if index != query_index:
                gallery_records.append(dict(record))

distractor_offset = max(all_crops) + 1
for offset, pid in enumerate(sorted(single_cam_ids)):
    distractor_pid = distractor_offset + offset
    for camname, records in sorted(all_crops[pid].items()):
        for record in records:
            staged = dict(record)
            staged["pid"] = distractor_pid
            gallery_records.append(staged)

camname_to_id = {}
for record in train_records + query_records + gallery_records:
    if record["camname"] not in camname_to_id:
        camname_to_id[record["camname"]] = len(camname_to_id)
    record["camid"] = camname_to_id[record["camname"]]

train_pid_map = {pid: index for index, pid in enumerate(sorted({record["pid"] for record in train_records}))}
for record in train_records:
    record["pid"] = train_pid_map[record["pid"]]



class ReIDImageDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(record["pid"]), int(record["camid"]), index


class RandomIdentitySampler(Sampler):
    def __init__(self, records, p, k):
        self.records = records
        self.p = p
        self.k = k
        self.batch_size = p * k
        self.pid_to_indices = defaultdict(list)
        for index, record in enumerate(records):
            self.pid_to_indices[int(record["pid"])] .append(index)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        batch_indices = []
        shuffled_pids = self.pids[:]
        random.shuffle(shuffled_pids)
        for pid in shuffled_pids:
            candidates = self.pid_to_indices[pid]
            if len(candidates) >= self.k:
                picked = random.sample(candidates, self.k)
            else:
                picked = random.choices(candidates, k=self.k)
            batch_indices.extend(picked)
            if len(batch_indices) >= self.batch_size:
                yield from batch_indices[: self.batch_size]
                batch_indices = batch_indices[self.batch_size :]

    def __len__(self):
        return len(self.pids) * self.k


H, W = CFG["img_size"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose(
    [
        T.Resize((H + 16, W + 16), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.Pad(10),
        T.RandomCrop((H, W)),
        T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomErasing(p=0.5, value="random"),
    ]
)
eval_transform = T.Compose(
    [
        T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

train_loader = DataLoader(
    ReIDImageDataset(train_records, train_transform),
    batch_size=TRAIN_BATCH_SIZE,
    sampler=RandomIdentitySampler(train_records, CFG["batch_p"], CFG["batch_k"]),
    num_workers=CFG["num_workers"],
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(
    ReIDImageDataset(query_records, eval_transform),
    batch_size=CFG["eval_batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)
gallery_loader = DataLoader(
    ReIDImageDataset(gallery_records, eval_transform),
    batch_size=CFG["eval_batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)


SPLIT_STATS = {
    "cityflow_root": str(CITYFLOW_ROOT),
    "raw_split_roots": [str(path) for path in RAW_SPLIT_ROOTS],
    "crop_dir": str(CROP_DIR),
    "train_images": len(train_records),
    "query_images": len(query_records),
    "gallery_images": len(gallery_records),
    "num_train_ids": len(train_pid_map),
    "num_eval_ids": len(eval_identity_ids),
    "num_single_camera_ids": len(single_cam_ids),
    "num_cameras": len(camname_to_id),
    "batch_size": TRAIN_BATCH_SIZE,
    "pk_sampler": {"p": CFG["batch_p"], "k": CFG["batch_k"]},
    "extraction": EXTRACTION_CFG,
}
print(json.dumps(SPLIT_STATS, indent=2))

In [ ]:
IBN_NET_URL = "https://github.com/XingangPan/IBN-Net/releases/download/v1.0/resnet101_ibn_a-59ea0ac6.pth"
IBN_WEIGHTS_PATH = OUTPUT_DIR / "resnet101_ibn_a_imagenet.pth"


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_val * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_val=0.0):
        super().__init__()
        self.lambda_val = float(lambda_val)

    def set_lambda(self, value):
        self.lambda_val = float(value)

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)


class IBN_a(nn.Module):
    def __init__(self, planes):
        super().__init__()
        half = planes // 2
        self.IN = nn.InstanceNorm2d(half, affine=True)
        self.BN = nn.BatchNorm2d(planes - half)

    def forward(self, x):
        split = x.shape[1] // 2
        return torch.cat([self.IN(x[:, :split]), self.BN(x[:, split:])], dim=1)


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


def load_ibn_imagenet_weights(base):
    state_dict = None
    if IBN_WEIGHTS_PATH.exists():
        print(f"Loading ImageNet weights from {IBN_WEIGHTS_PATH}")
        state_dict = torch.load(IBN_WEIGHTS_PATH, map_location="cpu")
    else:
        try:
            print(f"Downloading ImageNet weights from {IBN_NET_URL}")
            urllib.request.urlretrieve(IBN_NET_URL, IBN_WEIGHTS_PATH)
            state_dict = torch.load(IBN_WEIGHTS_PATH, map_location="cpu")
        except Exception as exc:
            print(f"Direct download failed: {exc}")
            print("Falling back to torch.hub.load_state_dict_from_url")
            state_dict = torch.hub.load_state_dict_from_url(
                IBN_NET_URL,
                map_location="cpu",
                progress=True,
            )
            torch.save(state_dict, IBN_WEIGHTS_PATH)

    load_result = base.load_state_dict(state_dict, strict=False)
    allowed_missing = {"fc.weight", "fc.bias"}
    if not set(load_result.missing_keys).issubset(allowed_missing):
        raise RuntimeError(f"Unexpected missing keys: {load_result.missing_keys}")
    if load_result.unexpected_keys:
        print(f"Unexpected pretrained keys ignored: {load_result.unexpected_keys}")


class ResNet101IBNDMT(nn.Module):
    def __init__(self, num_classes, num_cameras, gem_p=3.0, feat_dim=2048, pretrained=True, camera_aware=True):
        super().__init__()
        base = tv_models.resnet101(weights=None)
        for layer in [base.layer1, base.layer2, base.layer3]:
            for block in layer:
                if hasattr(block, "bn1"):
                    block.bn1 = IBN_a(block.bn1.num_features)
        for module in base.layer4.modules():
            if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                module.stride = (1, 1)

        if pretrained:
            load_ibn_imagenet_weights(base)

        self.backbone = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4,
        )
        self.pool = GeM(p=gem_p)
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1.0)
        nn.init.constant_(self.bottleneck.bias, 0.0)
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)
        self.feat_dim = feat_dim

        self.camera_aware = camera_aware
        self.num_cameras = int(num_cameras)
        if self.camera_aware:
            hidden_dim = feat_dim // 2
            self.grl = GradientReversalLayer(lambda_val=0.0)
            self.camera_head = nn.Sequential(
                nn.Linear(feat_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.3),
                nn.Linear(hidden_dim, self.num_cameras),
            )
            for module in self.camera_head.modules():
                if isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, std=0.001)
                    if module.bias is not None:
                        nn.init.constant_(module.bias, 0.0)
        else:
            self.grl = None
            self.camera_head = None

    def reset_classifier(self, num_classes):
        self.classifier = nn.Linear(self.feat_dim, num_classes, bias=False).to(next(self.parameters()).device)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward_features(self, x):
        x = self.backbone(x)
        global_feat = self.pool(x).view(x.size(0), -1)
        bn_feat = self.bottleneck(global_feat)
        return global_feat, bn_feat

    def forward(self, x, cam_ids=None):
        global_feat, bn_feat = self.forward_features(x)
        if self.training:
            logits = self.classifier(bn_feat)
            cam_logits = None
            if self.camera_aware and self.camera_head is not None and cam_ids is not None:
                cam_logits = self.camera_head(self.grl(global_feat))
            return logits, global_feat, bn_feat, cam_logits
        return F.normalize(bn_feat, p=2, dim=1)


def build_model(num_classes, num_cameras):
    model = ResNet101IBNDMT(
        num_classes=num_classes,
        num_cameras=num_cameras,
        gem_p=CFG["gem_p"],
        pretrained=True,
        camera_aware=True,
    )
    return model.to(device)


model = build_model(len(train_pid_map), len(camname_to_id))
print(
    json.dumps(
        {
            "model": "resnet101_ibn_a",
            "camera_aware": True,
            "num_cameras": len(camname_to_id),
            "params": sum(param.numel() for param in model.parameters()),
        },
        indent=2,
    )
)

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_one_hot = (1 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        return (-targets_one_hot * log_probs).sum(dim=1).mean()


class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, labels):
        distance = torch.cdist(feats, feats, p=2)
        is_pos = labels.unsqueeze(0).eq(labels.unsqueeze(1))
        is_neg = ~is_pos
        is_pos.fill_diagonal_(False)

        max_pos = torch.where(is_pos, distance, torch.zeros_like(distance)).max(dim=1)[0]
        min_neg = torch.where(is_neg, distance, torch.full_like(distance, 1e9)).min(dim=1)[0]
        valid = is_pos.any(dim=1) & is_neg.any(dim=1)
        if not valid.any():
            return feats.sum() * 0.0
        target = torch.ones_like(max_pos[valid])
        return self.ranking_loss(min_neg[valid], max_pos[valid], target)


def build_losses(num_classes):
    id_loss = CrossEntropyLabelSmooth(num_classes, epsilon=CFG["label_smoothing"]).to(device)
    triplet_loss = HardTripletLoss(margin=CFG["triplet_margin"]).to(device)
    camera_loss = nn.CrossEntropyLoss().to(device)
    return id_loss, triplet_loss, camera_loss


id_loss_fn, triplet_loss_fn, camera_loss_fn = build_losses(len(train_pid_map))
print(
    json.dumps(
        {
            "losses": ["cross_entropy_label_smooth", "hard_triplet", "camera_adversarial_grl"],
            "label_smoothing": CFG["label_smoothing"],
            "triplet_margin": CFG["triplet_margin"],
            "circle_weight": CFG["circle_weight"],
            "cam_loss_weight": CFG["cam_loss_weight"],
            "dmt_warmup_epochs": CFG["dmt_warmup_epochs"],
        },
        indent=2,
    )
)

In [ ]:
def build_optimizer(model_ref, base_lr):
    params = [
        {"params": model_ref.backbone.parameters(), "lr": base_lr * CFG["backbone_lr_factor"]},
        {"params": model_ref.pool.parameters(), "lr": base_lr},
        {"params": model_ref.bottleneck.parameters(), "lr": base_lr},
        {"params": model_ref.classifier.parameters(), "lr": base_lr},
    ]
    if model_ref.camera_aware and model_ref.camera_head is not None:
        params.append({"params": model_ref.camera_head.parameters(), "lr": base_lr})
    return torch.optim.AdamW(params, lr=base_lr, weight_decay=CFG["weight_decay"])


def build_scheduler(optimizer):
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=CFG["warmup_start_factor"],
        end_factor=1.0,
        total_iters=CFG["warmup_epochs"],
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(CFG["train_epochs"] - CFG["warmup_epochs"], 1),
        eta_min=CFG["eta_min"],
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[CFG["warmup_epochs"]],
    )


def get_grl_lambda(epoch):
    if epoch < CFG["dmt_warmup_epochs"]:
        return 0.0
    progress = (epoch - CFG["dmt_warmup_epochs"]) / max(CFG["train_epochs"] - CFG["dmt_warmup_epochs"], 1)
    return CFG["grl_lambda_max"] * min(progress, 1.0)


@torch.no_grad()
def extract_embeddings(model_ref, loader, flip=True):
    model_ref.eval()
    features = []
    pids = []
    camids = []
    indices = []
    for images, batch_pids, batch_camids, batch_indices in loader:
        images = images.to(device, non_blocking=True)
        embeddings = model_ref(images)
        if flip:
            embeddings = embeddings + model_ref(torch.flip(images, dims=[3]))
            embeddings = F.normalize(embeddings, p=2, dim=1)
        features.append(embeddings.cpu())
        pids.extend(batch_pids.tolist())
        camids.extend(batch_camids.tolist())
        indices.extend(batch_indices.tolist())
    return torch.cat(features, dim=0), np.asarray(pids), np.asarray(camids), np.asarray(indices)


def compute_cmc_map(query_features, query_pids, query_camids, gallery_features, gallery_pids, gallery_camids):
    distance = torch.cdist(query_features, gallery_features, p=2).cpu().numpy()
    indices = np.argsort(distance, axis=1)
    all_ap = []
    cmc = np.zeros(gallery_features.size(0), dtype=np.float64)
    valid_queries = 0

    for row in range(query_features.size(0)):
        order = indices[row]
        remove = (gallery_pids[order] == query_pids[row]) & (gallery_camids[order] == query_camids[row])
        keep = ~remove
        matches = (gallery_pids[order][keep] == query_pids[row]).astype(np.int32)
        if matches.sum() == 0:
            continue
        cumulative = np.cumsum(matches)
        precision = cumulative / (np.arange(matches.size) + 1)
        ap = (precision * matches).sum() / matches.sum()
        all_ap.append(ap)
        first_hit = np.where(matches == 1)[0][0]
        cmc[first_hit:] += 1
        valid_queries += 1

    if valid_queries == 0:
        return {"mAP": 0.0, "rank1": 0.0, "rank5": 0.0, "rank10": 0.0}

    cmc = cmc / valid_queries
    return {
        "mAP": float(np.mean(all_ap)),
        "rank1": float(cmc[0]) if cmc.size > 0 else 0.0,
        "rank5": float(cmc[4]) if cmc.size > 4 else 0.0,
        "rank10": float(cmc[9]) if cmc.size > 9 else 0.0,
    }


def evaluate_model(model_ref):
    qf, q_pids, q_camids, _ = extract_embeddings(model_ref, query_loader, flip=CFG["use_flip_eval"])
    gf, g_pids, g_camids, _ = extract_embeddings(model_ref, gallery_loader, flip=CFG["use_flip_eval"])
    return compute_cmc_map(qf, q_pids, q_camids, gf, g_pids, g_camids)


def train_one_epoch(model_ref, loader, optimizer, scaler, id_criterion, triplet_criterion, cam_criterion, epoch):
    model_ref.train()
    meter = defaultdict(float)
    cam_correct = 0
    cam_total = 0
    autocast_device = "cuda" if device.type == "cuda" else "cpu"

    for images, labels, camids, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        camids = camids.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=autocast_device, enabled=CFG["fp16"] and device.type == "cuda"):
            logits, global_feat, _, cam_logits = model_ref(images, cam_ids=camids)
            loss_id = id_criterion(logits, labels)
            loss_tri = triplet_criterion(global_feat, labels)
            loss = loss_id + loss_tri
            loss_cam = global_feat.new_tensor(0.0)
            if cam_logits is not None:
                cam_preds = cam_logits.argmax(dim=1)
                cam_correct += int((cam_preds == camids).sum().item())
                cam_total += int(camids.numel())
                if epoch >= CFG["dmt_warmup_epochs"]:
                    loss_cam = cam_criterion(cam_logits, camids)
                    loss = loss + CFG["cam_loss_weight"] * loss_cam

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_ref.parameters(), CFG["gradient_clip_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_ref.parameters(), CFG["gradient_clip_norm"])
            optimizer.step()

        meter["loss"] += float(loss.item())
        meter["id_loss"] += float(loss_id.item())
        meter["triplet_loss"] += float(loss_tri.item())
        meter["cam_loss"] += float(loss_cam.item())

    steps = max(len(loader), 1)
    metrics = {key: value / steps for key, value in meter.items()}
    metrics["cam_accuracy"] = float(cam_correct / max(cam_total, 1))
    return metrics


optimizer = build_optimizer(model, CFG["lr"])
scheduler = build_scheduler(optimizer)
scaler = torch.amp.GradScaler("cuda") if CFG["fp16"] and device.type == "cuda" else None
print(
    json.dumps(
        {
            "optimizer": "AdamW",
            "backbone_lr": CFG["lr"] * CFG["backbone_lr_factor"],
            "head_lr": CFG["lr"],
            "scheduler": "linear_warmup_then_cosine",
            "warmup_epochs": CFG["warmup_epochs"],
            "dmt_warmup_epochs": CFG["dmt_warmup_epochs"],
        },
        indent=2,
    )
)

STAGE1_HISTORY = []
STAGE1_BEST = {"mAP": -1.0, "epoch": -1, "path": str(CHECKPOINT_DIR / "stage1_best.pth")}

for epoch in range(CFG["train_epochs"]):
    grl_lambda = get_grl_lambda(epoch)
    if model.camera_aware and model.grl is not None:
        model.grl.set_lambda(grl_lambda)

    started = time.time()
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        id_loss_fn,
        triplet_loss_fn,
        camera_loss_fn,
        epoch,
    )
    scheduler.step()

    record = {
        "epoch": epoch + 1,
        "elapsed_sec": round(time.time() - started, 2),
        "backbone_lr": float(optimizer.param_groups[0]["lr"]),
        "head_lr": float(optimizer.param_groups[-1]["lr"]),
        "grl_lambda": float(grl_lambda),
        **{key: float(value) for key, value in train_metrics.items()},
    }

    if (epoch + 1) % CFG["eval_every"] == 0 or epoch == CFG["train_epochs"] - 1:
        eval_metrics = evaluate_model(model)
        record.update(eval_metrics)
        if eval_metrics["mAP"] > STAGE1_BEST["mAP"]:
            STAGE1_BEST.update({"mAP": eval_metrics["mAP"], "epoch": epoch + 1})
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "metrics": eval_metrics,
                    "config": CFG,
                },
                STAGE1_BEST["path"],
            )

    STAGE1_HISTORY.append(record)
    with (OUTPUT_DIR / "stage1_history.json").open("w", encoding="utf-8") as handle:
        json.dump(STAGE1_HISTORY, handle, indent=2)
    print(json.dumps(record, indent=2))

stage1_checkpoint = torch.load(STAGE1_BEST["path"], map_location="cpu")
model.load_state_dict(stage1_checkpoint["model"])
print(json.dumps(STAGE1_BEST, indent=2))

In [ ]:
# Stage 2 pseudo-label / UDA refinement: SKIPPED
# DMT in this notebook is the camera-adversarial GRL loss used during Stage 1.
# We intentionally do NOT run DBSCAN / mean-teacher refinement on CityFlowV2 itself,
# because findings show same-domain pseudo-label adaptation is harmful here.

print("Stage 2 pseudo-label/UDA skipped - using Stage 1 DMT checkpoint directly")
print(json.dumps(STAGE1_BEST, indent=2))

In [ ]:
FINAL_MODEL = model
FINAL_METRICS = evaluate_model(FINAL_MODEL)

print(json.dumps({"final_metrics": FINAL_METRICS, "stage1_best": STAGE1_BEST}, indent=2))

In [ ]:
BEST_MODEL_PATH = Path("/kaggle/working/resnet101ibn_dmt_cityflowv2_best.pth")
METADATA_PATH = Path("/kaggle/working/resnet101ibn_dmt_cityflowv2_metadata.json")
HISTORY_PATH = Path("/kaggle/working/resnet101ibn_dmt_cityflowv2_history.json")

torch.save(
    {
        "state_dict": FINAL_MODEL.state_dict(),
        "config": CFG,
        "metrics": FINAL_METRICS,
    },
    BEST_MODEL_PATH,
)

with HISTORY_PATH.open("w", encoding="utf-8") as handle:
    json.dump({"stage1": STAGE1_HISTORY}, handle, indent=2)

metadata = {
    "model": "resnet101_ibn_a",
    "recipe": "ResNet101-IBN-a DMT (ImageNet init, AdamW, ID + Triplet + camera-adversarial GRL, no circle, no DBSCAN/UDA stage)",
    "pooling": "GeM",
    "neck": "BNNeck",
    "pretraining": "ImageNet IBN-Net official weights",
    "dataset": "CityFlowV2 ReID",
    "img_size": list(CFG["img_size"]),
    "train_batch_size": TRAIN_BATCH_SIZE,
    "embedding_dim": model.feat_dim,
    "num_train_ids": len(train_pid_map),
    "num_cameras": len(camname_to_id),
    "resume_from_09d_checkpoint": False,
    "stage1_best": STAGE1_BEST,
    "final_metrics": FINAL_METRICS,
    "split_stats": SPLIT_STATS,
    "config": CFG,
}
with METADATA_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

print(
    json.dumps(
        {
            "best_model": str(BEST_MODEL_PATH),
            "metadata": str(METADATA_PATH),
            "history": str(HISTORY_PATH),
            "checkpoint_dir": str(CHECKPOINT_DIR),
        },
        indent=2,
    )
)